In [1]:
from python_scripts.horton_capsule import run_horton,load_horton_moments, run_psi4
import yaml

# Needed for formatting
from qcp_objects.objects.multipoles.molecular_multipole_moment import MolecularMultipoleMoments
from qcp_objects.objects.properties import rotate_multipoles

In [4]:
geom_file='data/h2o.xyz'
run_psi4(geom_file, method='pbe0',basis='aug-cc-pvtz')

# GET the moments file via the storage!!
# Check where are they centered (we want COM)
the_file='run_psi4/STORAGE_*.yaml'
import glob
storage_file=glob.glob(the_file)[0]
with open(storage_file,'r') as rd:
    store=yaml.safe_load(rd)
moms=store['results']['MolMoms_from_dens']
exp_center=store['results']['properties']['expansion_center']
fchk_file=store['files']['final_fchk']
import os
if not fchk_file.startswith('/'): fchk_file=os.path.join(os.path.dirname(storage_file),fchk_file)
assert os.path.isfile(fchk_file)

def format_file(moms,output_file):
    """ This creates a .mom file """

    moms=MolecularMultipoleMoments(moms, expansion_center=exp_center)
    molmom_file=moms.write_orient_moment_file(output_file=output_file)
    return molmom_file
molmom_file=os.path.join(os.path.dirname(storage_file), f"molmom.mom")
molmom_file=format_file(moms, output_file=molmom_file)

Started process with PID 69659


In [5]:
# Run horton
run_horton(f"{fchk_file}", method='LISA')
moments=load_horton_moments('horton_run')

Started process with PID 69703
